# Ablation 1: SSL Only (NO rebalancing) — Kaggle version, macro-F1 checkpoint selection

Self-contained: reruns SimCLR pre-training itself if a matching encoder checkpoint isn't already sitting in the working directory (Kaggle notebooks don't share files across separate runs the way Google Drive did).

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import os
import shutil
import random
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report
from collections import Counter
import json

# =====================================================================
# 1. SETUP & PATH CONFIGURATION (KAGGLE PATHS)
# =====================================================================
# Converted from the original Colab version, which mounted Google Drive
# and loaded a previously-trained SSL encoder from it. Kaggle notebooks
# don't share files across separate runs, so this notebook is
# self-contained: see section 2 below for the checkpoint fallback.

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

BACKUP_DIR = '/kaggle/input/datasets/ittisamurtunib/dataset/CSVs-20260711T054424Z-2-001/CSVs'
UNLABELED_DIR = '/kaggle/input/datasets/ittisamurtunib/dataset/ISIC_2019_Training_Input/ISIC_2019_Training_Input'

OUTPUT_DIR = '/kaggle/working/Thesis_Outputs'
LABELED_DIR = '/kaggle/working/data/labeled_real'

os.makedirs(LABELED_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

if os.path.exists(UNLABELED_DIR):
    unlabeled_count = len([f for f in os.listdir(UNLABELED_DIR) if f.endswith('.jpg')])
    print(f"Verified dataset. Found {unlabeled_count} images in Kaggle source input directory.")
else:
    raise FileNotFoundError(f"Could not locate image directory at {UNLABELED_DIR}")


Device: cuda
Verified dataset. Found 25331 images in Kaggle source input directory.


**Filter labeled images into the writable working directory**

In [2]:
train_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_train.csv'))
val_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_val.csv'))
test_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_test.csv'))
all_images = pd.concat([train_df, val_df, test_df])['image'].unique()

found = 0
for img_id in all_images:
    src = f'{UNLABELED_DIR}/{img_id}.jpg'
    dst = f'{LABELED_DIR}/{img_id}.jpg'
    if os.path.exists(src):
        if not os.path.exists(dst):
            shutil.copy(src, dst)
        found += 1

print(f"Copied/verified {found}/{len(all_images)} labeled images into working directory.")


Copied/verified 691/691 labeled images into working directory.


**SSL ENCODER: load if present, otherwise pre-train from scratch (self-contained)**

Same protocol as the main SSL+rebalance notebook: SimCLR, 20 epochs, batch 64, lr 3e-4, on the real unlabeled pool. This costs ~2 extra hours of compute on Kaggle's T4 if the checkpoint has to be retrained, but keeps this ablation fully reproducible on its own rather than silently depending on a checkpoint from a different notebook that Kaggle can't see.

In [3]:
class SimpleEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten()
        )
    def forward(self, x):
        return self.features(x)

class SSLClassifier(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.encoder(x))

class SimCLRTransform:
    def __init__(self, size=224):
        self.transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    def __call__(self, x):
        return self.transform(x), self.transform(x)

class UnlabeledDataset(Dataset):
    def __init__(self, image_ids, image_dir, transform):
        self.image_ids = image_ids
        self.image_dir = image_dir
        self.transform = transform
    def __len__(self):
        return len(self.image_ids)
    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img = Image.open(f'{self.image_dir}/{img_id}.jpg').convert('RGB')
        return self.transform(img)

class SimCLR(nn.Module):
    def __init__(self, encoder, projection_dim=64):
        super().__init__()
        self.encoder = encoder
        self.projector = nn.Sequential(
            nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, projection_dim)
        )
    def forward(self, x):
        h = self.encoder(x)
        z = F.normalize(self.projector(h), dim=1)
        return h, z

def nt_xent_loss(z_i, z_j, temperature=0.5):
    batch_size = z_i.shape[0]
    z = torch.cat([z_i, z_j], dim=0)
    sim_matrix = torch.mm(z, z.t()) / temperature
    mask = torch.eye(2 * batch_size, device=z.device).bool()
    sim_matrix = sim_matrix.masked_fill(mask, -9e15)
    pos_sim = torch.cat([
        torch.diag(sim_matrix, batch_size),
        torch.diag(sim_matrix, -batch_size)
    ])
    log_sum_exp = torch.logsumexp(sim_matrix, dim=1)
    loss = -(pos_sim - log_sum_exp)
    return loss.mean()

def pretrain_ssl_encoder(unlabeled_df_path, unlabeled_dir, out_path,
                          epochs=20, batch_size=64, lr=3e-4):
    """Reruns SimCLR pre-training from scratch, matching the main notebook's protocol."""
    unlabeled_df = pd.read_csv(unlabeled_df_path)
    unlabeled_ids = unlabeled_df['image'].tolist()
    available_ids = [i for i in unlabeled_ids if os.path.exists(f'{unlabeled_dir}/{i}.jpg')]
    print(f"Retraining SSL encoder on {len(available_ids)} unlabeled images "
          f"({epochs} epochs, batch {batch_size}, lr {lr})")

    enc = SimpleEncoder()
    simclr_model = SimCLR(enc).to(device)
    simclr_transform = SimCLRTransform()
    loader = DataLoader(
        UnlabeledDataset(available_ids, unlabeled_dir, simclr_transform),
        batch_size=batch_size, shuffle=True, drop_last=True, num_workers=2
    )
    opt = torch.optim.Adam(simclr_model.parameters(), lr=lr)

    for epoch in range(epochs):
        simclr_model.train()
        total_loss = 0
        pbar = tqdm(loader, desc=f"SSL pretrain epoch {epoch+1}/{epochs}")
        for view1, view2 in pbar:
            view1, view2 = view1.to(device), view2.to(device)
            _, z1 = simclr_model(view1)
            _, z2 = simclr_model(view2)
            loss = nt_xent_loss(z1, z2)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        print(f"Epoch {epoch+1}/{epochs}, Avg Contrastive Loss: {total_loss / len(loader):.4f}")

    torch.save(enc.state_dict(), out_path)
    print(f"SSL encoder retrained and saved to: {out_path}")
    return enc

## Checkpoint lookup: reuse an existing encoder if this Kaggle session
## already produced one (e.g. re-running this same notebook), otherwise
## retrain from scratch so this ablation has no external dependency.
ssl_ckpt_path = f'{OUTPUT_DIR}/ssl_encoder_real_24000.pth'

if os.path.exists(ssl_ckpt_path):
    encoder = SimpleEncoder()
    encoder.load_state_dict(torch.load(ssl_ckpt_path, map_location=device))
    print(f"Loaded existing SSL encoder from: {ssl_ckpt_path}")
else:
    print(f"No existing SSL encoder found at {ssl_ckpt_path} — retraining from scratch.")
    encoder = pretrain_ssl_encoder(
        unlabeled_df_path=os.path.join(BACKUP_DIR, 'unlabeled_pool_24k.csv'),
        unlabeled_dir=UNLABELED_DIR,
        out_path=ssl_ckpt_path,
        epochs=20, batch_size=64, lr=3e-4
    )

## Unfreeze all for fine-tuning
for param in encoder.parameters():
    param.requires_grad = True

model = SSLClassifier(encoder, 3).to(device)  ## 3 classes: BKL, MEL, NV


No existing SSL encoder found at /kaggle/working/Thesis_Outputs/ssl_encoder_real_24000.pth — retraining from scratch.
Retraining SSL encoder on 24000 unlabeled images (20 epochs, batch 64, lr 0.0003)


SSL pretrain epoch 1/20: 100%|██████████| 375/375 [06:59<00:00,  1.12s/it, loss=3.5913]


Epoch 1/20, Avg Contrastive Loss: 3.7119


SSL pretrain epoch 2/20: 100%|██████████| 375/375 [06:14<00:00,  1.00it/s, loss=3.3908]


Epoch 2/20, Avg Contrastive Loss: 3.4859


...

Epoch 18/20, Avg Contrastive Loss: 3.1799


SSL pretrain epoch 19/20: 100%|██████████| 375/375 [06:14<00:00,  1.00it/s, loss=3.1618]


Epoch 19/20, Avg Contrastive Loss: 3.1697


SSL pretrain epoch 20/20: 100%|██████████| 375/375 [06:13<00:00,  1.00it/s, loss=3.1316]

Epoch 20/20, Avg Contrastive Loss: 3.1651
SSL encoder retrained and saved to: /kaggle/working/Thesis_Outputs/ssl_encoder_real_24000.pth


**STANDARD DATASET (NO augmentation, NO oversampling)**

In [5]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class StandardDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
        self.classes = sorted(df['label'].unique())
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(f"{LABELED_DIR}/{row['image']}.jpg").convert('RGB')
        label = self.class_to_idx[row['label']]
        if self.transform:
            img = self.transform(img)
        return img, label

train_ds = StandardDataset(train_df, transform)
val_ds = StandardDataset(val_df, test_transform)
test_ds = StandardDataset(test_df, test_transform)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)
test_loader = DataLoader(test_ds, batch_size=16)

print(f"Train: {len(train_ds)} (original, no oversampling)")
print(f"Class distribution: {train_df['label'].value_counts().to_dict()}")


Train: 483 (original, no oversampling)
Class distribution: {'NV': 349, 'BKL': 99, 'MEL': 35}


**PLAIN CROSS-ENTROPY (NO class weights, NO Focal Loss)**

In [6]:
criterion = nn.CrossEntropyLoss()  ## Plain CE - no rebalancing
optimizer = optim.Adam(model.parameters(), lr=0.0001)

print("\nUsing Plain CrossEntropyLoss (No rebalancing)")



Using Plain CrossEntropyLoss (No rebalancing)


# **Training & Evaluation**

**CHECKPOINT-SELECTION FIX (per supervisor review):** the original ablation kept
whichever epoch had the highest plain validation *accuracy*, which lets a
majority-class-collapsed epoch (near-100% NV, 0% MEL) look artificially good.
This version selects the checkpoint by validation **macro-F1** instead, matching
the corrected criterion used elsewhere in the paper. Validation accuracy is
still logged every epoch for comparison, it is just no longer the selection
criterion. Kept single-seed (seed 42) to match the original ablation budget.

In [7]:
def train_epoch(model, loader):
    model.train()
    total_loss, correct = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return acc, macro_f1, all_labels, all_preds

print(f"\n{'='*36}")
print("ABLATION 1: SSL ONLY (NO rebalancing)")
print(f"{'='*36}")

epochs = 30
best_val_macro_f1 = -1.0
patience = 7
epochs_no_improve = 0
best_epoch = -1
ckpt_path = f'{OUTPUT_DIR}/best_ssl_only.pth'
history = {"train_loss": [], "train_acc": [], "val_acc": [], "val_macro_f1": []}

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader)
    val_acc, val_macro_f1, _, _ = evaluate(model, val_loader)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["val_macro_f1"].append(val_macro_f1)

    if val_macro_f1 > best_val_macro_f1:
        best_val_macro_f1 = val_macro_f1
        best_epoch = epoch + 1
        torch.save(model.state_dict(), ckpt_path)
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if (epoch + 1) % 3 == 0:
        print(f"Epoch {epoch+1}: Train={train_acc:.3f}, Val Acc={val_acc:.3f}, Val Macro-F1={val_macro_f1:.3f}")

    if epochs_no_improve >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

print(f"\nBest checkpoint: epoch {best_epoch} (Val Macro-F1={best_val_macro_f1:.3f})")

## Evaluate
model.load_state_dict(torch.load(ckpt_path, map_location=device))
test_acc, test_macro_f1, true_labels, pred_labels = evaluate(model, test_loader)

print("\n" + "="*36)
print("Results: SSL Only (NO rebalancing)")
print("="*36)
print(f"Test Accuracy: {test_acc:.3f}")
print(f"Test Macro-F1: {test_macro_f1:.3f}")

report = classification_report(true_labels, pred_labels,
                                target_names=train_ds.classes, output_dict=True)
print(classification_report(true_labels, pred_labels, target_names=train_ds.classes))

mel_recall = report['MEL']['recall']
print(f"\n===>>> MEL Recall: {mel_recall:.1%} <<<===")
print(f"Predictions: {Counter(pred_labels)}")

## Save
results = {
    'experiment': 'SSL_Only_No_Rebalance',
    'seed': SEED,
    'ssl': True,
    'rebalancing': False,
    'augmentation': False,
    'checkpoint_selection_metric': 'val_macro_f1',
    'best_epoch': best_epoch,
    'best_val_macro_f1': best_val_macro_f1,
    'test_accuracy': test_acc,
    'test_macro_f1': test_macro_f1,
    'mel_recall': mel_recall,
    'report': report,
    'history': history,
}
with open(f'{OUTPUT_DIR}/ssl_only_results.json', 'w') as f:
    json.dump(results, f, indent=2)

torch.save(model.state_dict(), f'{OUTPUT_DIR}/ssl_only_finetuned.pth')
print(f"\nSaved results and weights to {OUTPUT_DIR}")



ABLATION 1: SSL ONLY (NO rebalancing)
Epoch 3: Train=0.737, Val Acc=0.712, Val Macro-F1=0.277
Epoch 6: Train=0.741, Val Acc=0.731, Val Macro-F1=0.370
Epoch 9: Train=0.772, Val Acc=0.721, Val Macro-F1=0.376
Epoch 12: Train=0.766, Val Acc=0.731, Val Macro-F1=0.368
Epoch 15: Train=0.776, Val Acc=0.721, Val Macro-F1=0.376
Early stopping at epoch 15

Best checkpoint: epoch 8 (Val Macro-F1=0.394)

Results: SSL Only (NO rebalancing)
Test Accuracy: 0.740
Test Macro-F1: 0.392
              precision    recall  f1-score   support

         BKL       0.50      0.24      0.32        21
         MEL       0.00      0.00      0.00         8
          NV       0.77      0.96      0.85        75

    accuracy                           0.74       104
   macro avg       0.42      0.40      0.39       104
weighted avg       0.65      0.74      0.68       104


===>>> MEL Recall: 0.0% <<<===
Predictions: Counter({np.int64(2): 94, np.int64(0): 10})

Saved results and weights to /kaggle/working/Thesis_Outp

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m